# Streaming with gRPC and FIX (Concepts and Python Sketches)

This notebook is a **from-scratch to intermediate** guide to working with
**streaming APIs** in Python using:

- **gRPC** – modern, typed RPC over HTTP/2 (often for internal services).
- **FIX** – classic text-based trading protocol used by many brokers/venues.

The goal is to give you a **mental model** and **runnable shapes** of client
code that you can adapt to real endpoints later. We will not connect to any
live trading venues, but we will:

- Show how to write a Python gRPC client for a **server-streaming RPC**.
- Sketch how a Python FIX client looks when built with **QuickFIX**.
- Suggest public / demo services you can try once you are comfortable.


### Contents

- [1. Streaming in Trading Systems](#1-streaming-in-trading-systems)
- [2. gRPC Overview](#2-grpc-overview)
- [3. Defining a gRPC Price Stream Service](#3-defining-a-grpc-price-stream-service)
- [4. Python gRPC Client for a Server-Streaming RPC](#4-python-grpc-client-for-a-server-streaming-rpc)
- [5. Testing gRPC Against a Public Demo Service](#5-testing-grpc-against-a-public-demo-service)
- [6. FIX Overview](#6-fix-overview)
- [7. Python FIX Client Sketch with QuickFIX](#7-python-fix-client-sketch-with-quickfix)
- [8. Practical Patterns: Fan-In, Backpressure, and Translation](#8-practical-patterns-fan-in-backpressure-and-translation)


## 1. Streaming in Trading Systems

Two broad interaction styles:

- **Request/response**: you send a request, you get one response (HTTP REST,
  many RPC calls).
- **Streaming / push**: server continuously sends updates once you subscribe
  (market data, order updates).

gRPC and FIX both support streaming, but they target different worlds:

- **gRPC**: strongly-typed, protobuf-based, great for **internal services**
  and some public APIs.
- **FIX**: text tag=value protocol, de facto standard for **exchange/broker
  connectivity**, especially for orders + market data.

We will treat **gRPC** as the primary "how to connect" example, and **FIX** as
a conceptual overview with Python sketches.


## 2. gRPC Overview

gRPC basics:

- Messages are defined in a **.proto** file using **Protocol Buffers**.

- gRPC generates client and server code in multiple languages
  (here we use Python).
- Transport is **HTTP/2**, and gRPC supports several RPC styles:
  - Unary (one request, one response).
  - Server-streaming (one request, many responses).
  - Client-streaming (many requests, one response).
  - Bidirectional streaming (many ↔ many).

For market data and order updates we mainly care about
**server-streaming** and sometimes **bidirectional streaming** for
subscriptions.


## 3. Defining a gRPC Price Stream Service

A minimal price stream .proto might look like this:

```proto
syntax = "proto3";

package prices;

message PriceRequest {
  string symbol = 1;   // e.g. "BTCUSDT"
}

message PriceUpdate {
  string symbol = 1;
  double bid = 2;
  double ask = 3;
  int64 ts_micros = 4;
}

service PriceService {
  rpc StreamPrices(PriceRequest) returns (stream PriceUpdate);
}
```

To generate Python code from this, you would run (from a shell, once):

```bash
python -m pip install grpcio grpcio-tools
python -m grpc_tools.protoc \
  -I. \
  --python_out=. \
  --grpc_python_out=. \
  price_service.proto
```

This produces two modules (in the same directory):

- `price_service_pb2.py` – message classes.
- `price_service_pb2_grpc.py` – client and server stubs.

> In this notebook we assume these files exist, but the code is written in
> a way that it will **print helpful messages** if they do not.


## 4. Python gRPC Client for a Server-Streaming RPC

We will implement a small **async** client for `StreamPrices`. In real life you
would point this at your own gRPC server (possibly running in Docker or on
another machine).

Key ideas:

- Use `grpc.aio.insecure_channel` or `grpc.aio.secure_channel`.
- Use the generated `PriceServiceStub` from `price_service_pb2_grpc`.
- Call the server-streaming method and **iterate asynchronously** over
  responses.
- Wrap in a task with reconnect / cancellation logic.


In [1]:
# 4.1 Async gRPC client skeleton for StreamPrices

import asyncio
from typing import AsyncIterator

try:
    import grpc
    # Stubs generated from g_grpc-fix.proto (see Section 3)
    import g_grpc_fix_pb2 as price_pb2
    import g_grpc_fix_pb2_grpc as price_pb2_grpc
except ImportError:
    grpc = None
    print(
        "gRPC stubs not available. To try this for real, generate "
        "g_grpc_fix_pb2(.py) and g_grpc_fix_pb2_grpc(.py) from g_grpc-fix.proto "
        "as described in Section 3."
    )


async def stream_prices_once(symbol: str, target: str = "localhost:50051") -> None:
    """Connect once to a PriceService and stream updates for a symbol.

    This assumes you have a gRPC server implementing the service defined in
    price_service.proto running on `target`.
    """
    if grpc is None:
        print("grpc or stubs not available; skipping stream_prices_once.")
        return

    async with grpc.aio.insecure_channel(target) as channel:
        stub = price_pb2_grpc.PriceServiceStub(channel)
        request = price_pb2.PriceRequest(symbol=symbol)
        # Server-streaming RPC returns an async iterator
        async for update in stub.StreamPrices(request):
            print(
                f"{update.symbol} bid={update.bid:.5f} "
                f"ask={update.ask:.5f} ts={update.ts_micros}"
            )


# Demo skeleton: in a real environment you would run something like:
# asyncio.run(stream_prices_once("BTCUSDT", target="localhost:50051"))


## 5. Testing gRPC Against a Public Demo Service

There are several public gRPC test/demo services you can use to practice
without running your own server. Two examples:

- `grpcb.in` – a public gRPC testing service (see documentation at
  https://github.com/moul/grpcbin).
- `grpc-ecosystem/awesome-grpc` list – points to other public examples.

A very simple pattern is to:

- Inspect the service definition in the demo's documentation.
- Generate Python stubs from their `.proto` files.
- Adapt the `stream_prices_once` function to call their RPC instead
  (names will differ).

Because these services change occasionally and may require specific `.proto`
files, this notebook keeps the client example generic. The **workflow** to
practice with any gRPC service is:

1. Get the `.proto` file from the service documentation.
2. Run `grpc_tools.protoc` to generate Python code.
3. Import the generated stubs and write an async client as above.


## 6. FIX Overview

FIX (Financial Information eXchange) is a **text-based**, tag=value protocol used
heavily in trading. A single message might look like:

```text
8=FIX.4.4|9=112|35=D|49=CLIENT|56=BROKER|11=ORDER123|55=EUR/USD|54=1|38=100000|40=2|44=1.2345|10=128|
```

(delimiters are SOH bytes, shown here as `|` for readability).

Key concepts:

- **Session layer**: logon, heartbeats, logout, sequence numbers, resends.
- **Application layer**: orders, execution reports, market data messages.

You almost never implement FIX framing and resend logic by hand. Instead, you
use a FIX engine such as **QuickFIX** (C++ with Python bindings) or a broker's
SDK that wraps FIX internally.


## 7. Python FIX Client Sketch with QuickFIX

In Python, the most common way to talk FIX is via **QuickFIX** bindings:

```bash
python -m pip install quickfix
```

You configure a session via a `.cfg` file (initiator side). A minimal config
for a demo connection to a test gateway might look like:

```ini
[DEFAULT]
ConnectionType=initiator
BeginString=FIX.4.4
SenderCompID=CLIENT
TargetCompID=BROKER
HeartBtInt=30

[SESSION]
SocketConnectHost=127.0.0.1
SocketConnectPort=9877
FileStorePath=store
FileLogPath=log
UseDataDictionary=N
```

> Many brokers / test environments provide a sample QuickFIX config. The
> above is just a shape; you must adapt it to real credentials.


In [ ]:
# 7.1 QuickFIX Application skeleton (illustrative only)

try:
    import quickfix as fix
except ImportError:
    fix = None
    print("QuickFIX not installed; run 'pip install quickfix' to try this.")


if fix is not None:
    class Application(fix.Application):
        """Minimal QuickFIX Application subclass.

        Override callbacks to react to session and application messages.
        """

        def onCreate(self, sessionID):
            print("Session created:", sessionID)

        def onLogon(self, sessionID):
            print("Logon:", sessionID)

        def onLogout(self, sessionID):
            print("Logout:", sessionID)

        def toAdmin(self, message, sessionID):
            # Outgoing admin messages (Logon, Heartbeat, etc.)
            pass

        def fromAdmin(self, message, sessionID):
            # Incoming admin messages
            pass

        def toApp(self, message, sessionID):
            # Outgoing application messages (e.g. NewOrderSingle)
            pass

        def fromApp(self, message, sessionID):
            # Incoming application messages (e.g. ExecutionReport, MarketData)
            print("FromApp:", message)


    def run_fix_initiator(cfg_path: str) -> None:
        """Start a QuickFIX initiator using the given config file.

        This keeps running until you kill the process. In a Jupyter notebook
        you typically run this in a separate terminal or use threads.
        """
        settings = fix.SessionSettings(cfg_path)
        app = Application()
        store_factory = fix.FileStoreFactory(settings)
        log_factory = fix.FileLogFactory(settings)
        initiator = fix.SocketInitiator(app, store_factory, settings, log_factory)
        initiator.start()
        print("FIX initiator started. Press Ctrl+C to stop (in a real script).")
        # In a script you would block here; in a notebook be cautious.
        # time.sleep(...) or wait on some event.

    # Demo shape (do not run blindly in notebook):
    # run_fix_initiator("quickfix_client.cfg")


## 8. Practical Patterns: Fan-In, Backpressure, and Translation

Common patterns when working with gRPC/FIX streams:

- **Fan-in**: multiple upstream streams (several gRPC subscriptions or FIX
  sessions) feeding into a single internal queue.
- **Backpressure**: if your processing code is slower than the stream, you
  need to buffer (bounded) or drop/throttle.
- **Translation**: convert wire-level messages (protobuf or FIX fields) into
  domain objects (e.g. dataclasses) as early as possible.

Below is a simplified fan-in example using an `asyncio.Queue` and a dummy
producer to simulate a stream.


In [ ]:
# 8.1 Fan-in pattern with asyncio.Queue (simulated streams)

import asyncio
from dataclasses import dataclass
from typing import Any

@dataclass
class PriceEvent:
    source: str
    symbol: str
    bid: float
    ask: float


async def fake_stream(source: str, symbol: str, out_queue: asyncio.Queue[PriceEvent]) -> None:
    """Simulate a price stream by periodically putting events into a queue."""
    for i in range(5):
        bid = 100.0 + i
        ask = bid + 0.1
        await out_queue.put(PriceEvent(source, symbol, bid, ask))
        await asyncio.sleep(0.1)


async def consumer(queue: asyncio.Queue[PriceEvent]) -> None:
    while True:
        event = await queue.get()
        print(f"[{event.source}] {event.symbol} {event.bid:.2f}/{event.ask:.2f}")
        queue.task_done()


async def main_fan_in() -> None:
    q: asyncio.Queue[PriceEvent] = asyncio.Queue(maxsize=100)
    # In a real system, these would be tasks consuming gRPC or FIX streams.
    producers = [
        asyncio.create_task(fake_stream("GRPC", "BTCUSDT", q)),
        asyncio.create_task(fake_stream("FIX", "BTCUSDT", q)),
    ]
    cons = asyncio.create_task(consumer(q))
    await asyncio.gather(*producers)
    await q.join()
    cons.cancel()


# Demo skeleton:
# asyncio.run(main_fan_in())
